# 第 2 章互动实验：卷积、梯度与边缘

本实验只用 OpenCV 作为观察基线，不给出 `2-1_手写相关运算.py` 的实现。数据由代码合成，不依赖网络。先预测噪声、平滑尺度和阈值如何改变边缘，再拖动滑块验证。

In [ ]:
%matplotlib inline
import importlib.util
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Heiti SC', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError('请在课程目录内启动 Jupyter')


CHAPTER = find_chapter('02_卷积与边缘')


def load_homework(filename):
    """读入本章的一个作业文件。每次调用都重新读，改完作业后重新运行单元就能看到新结果。
    设置环境变量 COURSE_IMPL_DIR 可以换成另一份实现（维护者验证用）。"""
    override = os.environ.get('COURSE_IMPL_DIR')
    base = CHAPTER
    if override:
        base = Path(override) / CHAPTER.name
        if not base.is_dir():                      # 相对路径：按课程根目录再找一次
            base = CHAPTER.parent / override / CHAPTER.name
    spec = importlib.util.spec_from_file_location('homework_' + filename[:3].replace('-', '_'), base / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        result = fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f'先完成作业里的 {fn.__name__}：{exc}')
        return None
    if result is None:
        print(f'先完成作业里的 {fn.__name__}（现在返回的是 None）')
    return result


print('OpenCV:', cv2.__version__)

In [ ]:
def make_scene(noise_std=10):
    image = np.full((128, 160), 35, dtype=np.float32)
    image[28:100, 35:125] = 170
    cv2.line(image, (12, 112), (148, 18), 235, 2)
    rng = np.random.default_rng(2026)
    image += rng.normal(0, noise_std, image.shape)
    return np.clip(image, 0, 255).astype(np.uint8)

def show_edges(noise_std=10, sigma=1.0, low=50, high=120):
    image = make_scene(noise_std)
    k = max(3, int(np.ceil(6 * sigma)) | 1)
    smooth = cv2.GaussianBlur(image, (k, k), sigma)
    gx = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(gx, gy)
    edges = cv2.Canny(smooth, low, max(high, low + 1), L2gradient=True)
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
    for ax, data, title, cmap in zip(
        axes, [image, gx, magnitude, edges],
        ['合成图', '带符号 Gx', '梯度幅值', f'Canny：{np.count_nonzero(edges)} 点'],
        ['gray', 'coolwarm', 'magma', 'gray']):
        ax.imshow(data, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_edges,
    noise_std=IntSlider(min=0, max=40, step=5, value=10),
    sigma=FloatSlider(min=0.5, max=3.0, step=0.5, value=1.0),
    low=IntSlider(min=10, max=150, step=10, value=50),
    high=IntSlider(min=40, max=240, step=10, value=120),
);

## 用你的 `correlate2d` 做三个实验

下面三个实验对应讲义 2.1 节和 2.3 节新增的实测内容。每个实验先写下预测，再运行。

三个单元都会调用你在 `2-1_手写相关运算.py` 里写的 `correlate2d`。作业还没完成时，单元只打印一行“先完成作业里的 correlate2d”，OpenCV 的对照结果照常显示，不会报错；完成作业后重新运行单元即可，不用重启内核。

### 实验 A：冲激图上印出来的是什么（讲义 2.1 节，图 1）

一张只有正中间一个像素为 1 的 7×7 图，核是 `[[0,1,2],[3,4,5],[6,7,8]]`。

先预测：

- `cv2.filter2D` 在冲激周围印出来的 3×3 是核本身，还是核上下左右翻转后的样子？
- 严格的卷积印出来的是哪一个？
- 你的 `correlate2d` 应该和谁一致？

In [ ]:
impulse = np.zeros((7, 7), dtype=np.float32)
impulse[3, 3] = 1
K = np.arange(9, dtype=np.float32).reshape(3, 3)
correlate2d = load_homework('2-1_手写相关运算.py').correlate2d

print('核 K：')
print(K.astype(int))
print('cv2.filter2D(impulse, -1, K) 在冲激周围的 3×3：')
print(cv2.filter2D(impulse, cv2.CV_32F, K)[2:5, 2:5].astype(int))
print('把 K 上下左右翻转后再交给 filter2D（严格卷积）：')
print(cv2.filter2D(impulse, cv2.CV_32F, K[::-1, ::-1].copy())[2:5, 2:5].astype(int))
mine = attempt(correlate2d, impulse, K)
if mine is not None:
    print('你的 correlate2d：')
    print(np.round(mine[2:5, 2:5]).astype(int))
    same = np.allclose(mine, cv2.filter2D(impulse, cv2.CV_32F, K), atol=1e-4)
    print('和 cv2.filter2D 一致' if same else '和 cv2.filter2D 不一致：对照上面两个矩阵，看是不是把核翻转了')

### 实验 B：四种边界填充在最左一列给出什么（讲义 2.3 节，图 4）

一张每往右一列亮 2 的斜坡图，最左一列的灰度是 100。图像内部的 Sobel x 响应处处是 16。

先预测最左一列的响应：

- 补零（`BORDER_CONSTANT`）
- 复制边缘（`BORDER_REPLICATE`）
- 镜像、边界像素重复（`BORDER_REFLECT`，对应 `np.pad` 的 `symmetric`）
- 镜像、边界像素不重复（`BORDER_REFLECT_101`，对应 `np.pad` 的 `reflect`，作业要求的就是这一种）

四种里有没有哪一种能还原出内部的 16？你的 `correlate2d` 应该和哪一行相同？

In [ ]:
ramp = np.tile(100 + 2 * np.arange(16, dtype=np.float32), (8, 1))
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
borders = {
    '补零 BORDER_CONSTANT': cv2.BORDER_CONSTANT,
    '复制边缘 BORDER_REPLICATE': cv2.BORDER_REPLICATE,
    '镜像、重复边界 BORDER_REFLECT': cv2.BORDER_REFLECT,
    '镜像、不重复边界 BORDER_REFLECT_101': cv2.BORDER_REFLECT_101,
}
print(f"{'填充方式':<34}{'最左一列':>8}{'第 2 列':>8}{'内部':>8}")
for name, border in borders.items():
    response = cv2.filter2D(ramp, cv2.CV_32F, sobel_x, borderType=border)
    print(f'{name:<36}{response[4, 0]:>10.0f}{response[4, 1]:>10.0f}{response[4, 8]:>10.0f}')

correlate2d = load_homework('2-1_手写相关运算.py').correlate2d
mine = attempt(correlate2d, ramp, sobel_x)
if mine is not None:
    print(f"{'你的 correlate2d':<36}{mine[4, 0]:>10.0f}{mine[4, 1]:>10.0f}{mine[4, 8]:>10.0f}")
print('边界外的像素是编出来的，四种填充只是四种不同的猜法。靠边一圈的梯度不要拿来做测量。')

### 实验 C：一次二维和两次一维（讲义 2.1 节“可分离卷积”）

7×7 的高斯核可以写成一个列向量乘一个行向量。用你的 `correlate2d` 分别做“一次 7×7”和“先 1×7 再 7×1”。这个实验顺便检查你的函数能不能处理不是方形的核。

先预测：

- 两种做法的结果差多少？包括图像最外面几圈吗？
- 每个输出像素各要做多少次乘法？
- Sobel x、4 邻域的 Laplacian 能不能这样拆？判断的依据是核的什么性质？

In [ ]:
correlate2d = load_homework('2-1_手写相关运算.py').correlate2d
rng = np.random.default_rng(7)
texture = rng.uniform(0, 255, (40, 56)).astype(np.float32)
x = np.arange(-3, 4, dtype=np.float32)
g = np.exp(-x ** 2 / (2 * 1.2 ** 2))
g /= g.sum()
gauss_2d = np.outer(g, g).astype(np.float32)

once = attempt(correlate2d, texture, gauss_2d)
if once is not None:
    rows = attempt(correlate2d, texture, g.reshape(1, 7))
    twice = attempt(correlate2d, rows, g.reshape(7, 1))
    print('一次 7×7 的输出：', once.shape, once.dtype, '｜两次一维的输出：', twice.shape, twice.dtype)
    print(f'两种做法的最大差：{np.abs(once - twice).max():.2e}（灰度范围 0 到 255，float32 的舍入误差在 1e-4 量级）')
    print('每个像素的乘法次数：一次二维 7×7 = 49 次，两次一维 7 + 7 = 14 次')

kernels = {
    '高斯 7×7': gauss_2d,
    'Sobel x': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
    'Laplacian（4 邻域）': np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32),
}
for name, kernel in kernels.items():
    rank = np.linalg.matrix_rank(kernel, tol=1e-6)
    print(f'{name}：秩 {rank}，', '能拆成一列乘一行' if rank == 1 else '不能拆成一列乘一行')

## 破坏实验与记录

1. 把噪声调到 40 且平滑降到 0.5，记录误检。
2. 把平滑提高到 3.0，观察细斜线定位和连续性。
3. 让低阈值接近高阈值，观察弱边是否断裂。

把参数、边缘像素数、失败原因写入 `实验记录.md`，再独立完成作业。